### What we are doing
We are taking pretrained BERT --> Training it on our dataset

    Text → Tokenizer → BERT → Classification Head → Output

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
#sample dataset

texts = [
    "I love this product",
    "This is terrible",
    "Amazing experience",
    "Worst purchase ever"
] 

labels = [1, 0, 1, 0] #1: positive, 0: negative

In [3]:
#load tokenizer 
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=16,
            return_tensors='pt'
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels']= torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [5]:
#dataloader
dataset = TextDataset(texts, labels)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [6]:
#loading model
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


This adds a classification head on top of BERT

In [7]:
#optimizer 
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [8]:
#Training Loop
model.train()

for epoch in range(3):
    for batch in loader:

        outputs = model(
            input_ids = batch['input_ids'],
            attention_mask = batch['attention_mask'],
            labels = batch['labels']
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

    print(f'Epoch: {epoch} Loss: {loss.item()}')

Epoch: 0 Loss: 0.6971577405929565
Epoch: 1 Loss: 0.6826048493385315
Epoch: 2 Loss: 0.7709102630615234


In [9]:
model.eval()

text = "I really love this!"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1)

print(prediction)  # 1 = positive

tensor([1])


Concepts used:

1. AutoModelForSequenceClassification

       Instead of raw embeddings, it adds:
    
        BERT → Linear layer → Output classes
2. labels = torch.long; because CrossEntropyLoass is used internally
3. Instead of writing:

        * model
        * loss
        * forward pass logic
        
        You just do:
        
        outputs = model(..., labels=...)
        loss = outputs.loss